In [1]:
import json
import re
import fitz  # PyMuPDF
from typing import List, Dict, Optional
from langchain_text_splitters import RecursiveCharacterTextSplitter


# =========================
# 1. PDF EXTRACTION
# =========================
def extract_text_with_fitz(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = [page.get_text("text") for page in doc]
    return "\n".join(pages)


# =========================
# 2. CLEANING (ĐÃ CẢI TIẾN)
# =========================
def clean_legal_pdf_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove Công báo header/footer
    text = re.sub(r"CÔNG BÁO/Số\s+\d+\s*\+\s*\d+/Ngày\s+\d{1,2}-\d{1,2}-\d{4}", " ", text, flags=re.IGNORECASE)
    
    # Remove isolated page numbers
    text = re.sub(r"\n\s*\d{1,3}\s*\n", "\n", text)
    
    # Remove footnote kiểu "¹ Luật An ninh mạng...", "² Cụm từ...", "⁴ Cụm từ..."
    text = re.sub(r"\n?\s*\d{0,2}[¹²³⁴⁵⁶⁷⁸⁹⁰]+\s*Cụm từ.+?(?=\n[A-ZĐ]|Điều\s|\Z)", "", text, flags=re.DOTALL)
    text = re.sub(r"\n?\s*\d+\s*Luật An ninh mạng.+?(?=\n[A-ZĐ]|Điều\s|\Z)", "", text, flags=re.DOTALL)

    # Clean excessive spaces and newlines
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# =========================
# 3. PARSING CHAPTERS + ARTICLES
# =========================
CHAPTER_REGEX = re.compile(r"(Chương\s+[IVXLCDM]+)\s*\n+\s*([^\n]+)", flags=re.UNICODE)
ARTICLE_REGEX = re.compile(r"(Điều\s+\d+[a-z]?\.\s*[^\n]+)", flags=re.UNICODE)


def parse_chapters(text: str) -> List[Dict]:
    matches = list(CHAPTER_REGEX.finditer(text))
    chapters = []

    if not matches:
        return [{"chapter_label": None, "chapter_title": None, "chapter_full": None, "content": text}]

    for i, match in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chapter_label = match.group(1).strip()
        chapter_title = match.group(2).strip()
        chapter_full = f"{chapter_label} {chapter_title}"
        content = text[match.end():end].strip()

        chapters.append({
            "chapter_label": chapter_label,
            "chapter_title": chapter_title,
            "chapter_full": chapter_full,
            "content": content
        })
    return chapters


def split_article_blocks(chapter_content: str) -> List[Dict]:
    matches = list(ARTICLE_REGEX.finditer(chapter_content))
    articles = []

    if not matches:
        return []

    for i, match in enumerate(matches):
        article_title = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(chapter_content)
        article_content = chapter_content[start:end].strip()

        articles.append({
            "article_title": article_title,
            "article_content": article_content
        })
    return articles


# =========================
# 4. SPLIT LONG ARTICLES (ĐÃ SỬA - GIỮ NGUYÊN ĐIỀU)
# =========================
def split_long_article(article_text: str, max_length: int = 2800) -> List[str]:
    """
    Ưu tiên giữ nguyên toàn bộ 1 Điều làm 1 chunk.
    Chỉ tách khi Điều thực sự quá dài (> 2800 ký tự).
    """
    if len(article_text) <= max_length:
        return [article_text.strip()]

    # Nếu quá dài, tách theo nhóm lớn (không tách từng khoản 1., 2., 3...)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_length,
        chunk_overlap=250,
        separators=["\n\n\n", "\n\n", "\nĐiều ", "\n\n", "\n", ". ", " ", ""]
    )
    return splitter.split_text(article_text)


# =========================
# 5. FULL PROCESSING (ĐÃ SỬA)
# =========================
def process_legal_pdf(
    pdf_path: str,
    source_file: str,
    law_name: Optional[str] = None,
    domain: Optional[str] = None,
    effective_date: Optional[str] = None,
    law_id: Optional[str] = None,
    max_context_len: int = 2800          # TĂNG LÊN 2800
) -> List[Dict]:
    
    raw_text = extract_text_with_fitz(pdf_path)
    cleaned_text = clean_legal_pdf_text(raw_text)

    chapters = parse_chapters(cleaned_text)
    final_chunks = []

    for chapter in chapters:
        chapter_full = chapter["chapter_full"]
        articles = split_article_blocks(chapter["content"])

        for article in articles:
            article_title = article["article_title"]
            article_content = article["article_content"].strip()

            if not article_content:
                continue

            combined_title = f"{article_title} | {chapter_full}" if chapter_full else article_title

            base_payload = {
                "law_name": law_name,
                "source_file": source_file,
                "domain": domain,
                "effective_date": effective_date,
                "law_id": law_id,
                "chapter": chapter_full,
                "article": article_title,
                "title": combined_title
            }

            # === PHẦN CHÍNH: Giữ nguyên 1 Điều ===
            sub_parts = split_long_article(article_content, max_context_len)

            for idx, part in enumerate(sub_parts, start=1):
                chunk = {
                    **base_payload,
                    "context": part.strip(),
                }
                # Chỉ thêm sub_part khi thực sự tách nhiều phần
                if len(sub_parts) > 1:
                    chunk["sub_part"] = idx
                final_chunks.append(chunk)

    return final_chunks


# =========================
# 6. SAVE OUTPUT
# =========================
def save_to_json(data: List[Dict], output_file: str):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# =========================
# 7. MAIN
# =========================
if __name__ == "__main__":
    pdf_path = r"E:\VN-legal-chatbot\data_processing\Luat_tieu_dung.pdf"   # dùng raw string
    source_file = "Luat_BVQLNTD.pdf"
    law_name = "Luật Bảo vệ quyền lợi người tiêu dùng 2023"
    domain = "tieu_dung"
    effective_date = "2024-07-01"
    law_id = "BVQLNTD_2023"
    output_json = "output_qdrant_ready_luat_tieu_dung_new.json"

    chunks = process_legal_pdf(
        pdf_path=pdf_path,
        source_file=source_file,
        law_name=law_name,
        domain=domain,
        effective_date=effective_date,
        law_id=law_id,
        max_context_len=2800
    )

    print(f"Total chunks created: {len(chunks)}")
    if chunks:
        print("Sample chunk:")
        print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

    save_to_json(chunks, output_json)
    print(f"Saved to {output_json}")

d:\neit_ng\prjs_i\py_cmm_4thy2nds\nlp_thuchanh\prj_vn_legal_chatbot\conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks created: 88
Sample chunk:
{
  "law_name": "Luật Bảo vệ quyền lợi người tiêu dùng 2023",
  "source_file": "Luat_tieu_dung.pdf",
  "domain": "tieu_dung",
  "effective_date": "2024-07-01",
  "law_id": "BVQLNTD_2023",
  "chapter": "Chương I NHỮNG QUY ĐỊNH CHUNG",
  "article": "Điều 1. Phạm vi điều chỉnh",
  "title": "Điều 1. Phạm vi điều chỉnh | Chương I NHỮNG QUY ĐỊNH CHUNG",
  "context": "Luật này quy định về nguyên tắc, chính sách bảo vệ quyền lợi người tiêu \ndùng; quyền và nghĩa vụ của người tiêu dùng; trách nhiệm của tổ chức, cá nhân \nkinh doanh đối với người tiêu dùng; hoạt động bảo vệ quyền lợi người tiêu dùng \ncủa cơ quan, tổ chức; giải quyết tranh chấp giữa người tiêu dùng và tổ chức, cá \nnhân kinh doanh; quản lý nhà nước về bảo vệ quyền lợi người tiêu dùng."
}
Saved to output_qdrant_ready_luat_tieu_dung_new.json
